# Lab 1 · Exercise A — Gaussian, toy data, fits

**Goals:** build a pdf, generate toy data, fit by maximum likelihood (unbinned and binned), plot.
Fill every cell marked `TODO`, run the notebook top to bottom, and answer the three questions at the end.
Reference: RooFit tutorials `rf101_basics`, `rf201`, `rf202`.

In [ ]:
import ROOT
%jsroot on   # interactive drawing in the notebook (needed on Binder)
import numpy as np

MATRICOLA = 0   # TODO: write your student ID number here (digits only)
assert MATRICOLA != 0, "Set MATRICOLA first: it seeds YOUR random toy data"

ROOT.RooRandom.randomGenerator().SetSeed(MATRICOLA)
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.WARNING)

def check(name, ok):
    """Minimal self-check: prints PASS/FAIL and stops the notebook on FAIL."""
    print(("PASS  " if ok else "FAIL  ") + name)
    assert ok, name

## A1 · Unbinned fit of a Gaussian
Gaussian with mean = 0 and σ = 1. Plot it. Change σ to 3. Generate 10 000 **unbinned** events, fit by ML, plot.

In [ ]:
x     = ROOT.RooRealVar("x", "x", -10, 10)
mean  = ROOT.RooRealVar("mean", "mean", 0, -10, 10)
sigma = ROOT.RooRealVar("sigma", "sigma", 1, 0.1, 10)    # start with sigma = 1

gauss = None   # TODO: build the Gaussian pdf from x, mean, sigma  (RooGaussian)

frame0 = x.frame(Title="Gaussian pdf: sigma = 1 and sigma = 3")
# TODO: plot the pdf on frame0 with sigma = 1, then set sigma to 3 (sigma.setVal(3)) and plot again
c0 = ROOT.TCanvas("c0", "c0", 800, 500)
frame0.Draw()
c0.Draw()

In [ ]:
# sigma is now 3: generate and fit
data   = None   # TODO: generate 10000 unbinned events from gauss (observable x)
result = None   # TODO: fit gauss to data by maximum likelihood; keep the fit result (Save=True, PrintLevel=-1)

frame1 = x.frame(Title="Unbinned ML fit")
# TODO: plot data first, then the pdf
c1 = ROOT.TCanvas("c1", "c1", 800, 500)
frame1.Draw()
c1.Draw()
mean.Print(); sigma.Print()

In [ ]:
check("pdf and data exist", gauss is not None and data is not None and result is not None)
check("fit converged", result.status() == 0)
check("sigma compatible with truth (3)", abs(sigma.getVal() - 3) < 4 * sigma.getError())
check("mean compatible with truth (0)", abs(mean.getVal() - 0) < 4 * mean.getError())

## A2 · Binned fit of an exponential
Exponential pdf with rate = −1/τ, τ = 4.5 (mean life). Generate a **binned** dataset of 1000 events,
bin width 0.5. Fit by ML and plot.
*Tip:* express the rate with a `RooFormulaVar("rate", "-1./@0", ROOT.RooArgList(tau))`;
set the number of bins with `t.setBins(...)` **before** `generateBinned`.

In [ ]:
t    = ROOT.RooRealVar("t", "t", 0, 10)
tau  = None                # TODO: RooRealVar tau, start 4.5, range 0.5 .. 10
rate = None                # TODO: RooFormulaVar for -1/tau
expo = None                # TODO: RooExponential(t, rate)

t.setBins(20)              # bin width 0.5
data_b = None              # TODO: generate a BINNED dataset of 1000 events
# start the fit from a wrong value, so the fit has some work to do:
# tau.setVal(2.0)
res_b = None               # TODO: fit expo to data_b (Save=True, PrintLevel=-1)

frame2 = t.frame(Title="Binned ML fit")
# TODO: plot data_b, then the pdf
c2 = ROOT.TCanvas("c2", "c2", 800, 500)
frame2.Draw()
c2.Draw()
tau.Print()

In [ ]:
check("binned dataset built", data_b is not None and res_b is not None)
check("fit converged", res_b.status() == 0)
check("tau compatible with truth (4.5)", abs(tau.getVal() - 4.5) < 4 * tau.getError())

## A3 · (optional) Ranges of a parameter and of an observable
Refit the Gaussian of A1 in two modified settings and compare **best values and errors** with A1:
(a) a **parameter limit** that excludes the true value, `sigma.setRange(0.1, 2)`; (b) a **sub-range of the observable**, `x.setRange("fit", -5, 5)` and `fitTo(..., Range="fit")`.
Look at the printout of (a): where did sigma stop, and what is its error?

In [ ]:
# A3 (a): a parameter limit that excludes the true value (sigma = 3)
ref = (mean.getVal(), mean.getError(), sigma.getVal(), sigma.getError())          # best values of A1
print("A1            : mean = %.3f +- %.3f   sigma = %.3f +- %.3f" % ref)
sigma.setRange(0.1, 2); sigma.setVal(1); mean.setVal(0)
res_a = gauss.fitTo(data, Save=True, PrintLevel=-1)
print("sigma < 2     : mean = %.3f +- %.3f   sigma = %.3f +- %.3f" % (mean.getVal(), mean.getError(), sigma.getVal(), sigma.getError()))

# A3 (b): fit only a sub-range of the observable x
sigma.setRange(0.1, 10); sigma.setVal(1); mean.setVal(0)
x.setRange("fit", -5, 5)
res_b = gauss.fitTo(data, Range="fit", Save=True, PrintLevel=-1)
print("x in [-5, 5]  : mean = %.3f +- %.3f   sigma = %.3f +- %.3f" % (mean.getVal(), mean.getError(), sigma.getVal(), sigma.getError()))

## Questions (answer in this cell, 2–3 lines each)
1. Are the fitted values and errors compatible with the true ones? How did you decide?
2. Repeat A1 with 100 events. What happens to the errors? Is the scaling what you expect?
3. Why is the error on σ different from the error on the mean?
4. (A3, optional) What changed in the best values and in the errors in the two cases, and why?

In [ ]:
# Question 2: repeat A1 with only 100 events and compare the errors
sigma.setVal(3); mean.setVal(0)
data_small = None   # TODO: 100 events
# TODO: fit it and print the errors of mean and sigma, together with those from the 10000-event fit